# Task 2 — Generate Product Descriptions

For each of the 50 products in the dataset, we call **Gemma-2-9b-it** via Nebius Token Factory to generate a persuasive 50–90 word description. We collect the generated text, latency, and token counts, then save everything to `assignment_01.xlsx` with blank evaluation columns.

In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from shared.config import (
    DATASET_PATH,
    GENERATION_MODEL,
    NEBIUS_BASE_URL,
    OUTPUT_EXCEL,
)
from shared.constants import CRITERIA, SYSTEM_PROMPT
from shared.utils import build_user_message

client = OpenAI(
    base_url=NEBIUS_BASE_URL,
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

In [ ]:
df = pd.read_excel(DATASET_PATH)
print(f"{len(df)} products loaded")
df.head(3)

## System Prompt

The prompt is designed to:
- Set a clear role (e-commerce copywriter)
- State the length constraint explicitly (50–90 words)
- Require grounding — only use the provided product info
- Request a persuasive, friendly tone

In [ ]:
print(SYSTEM_PROMPT)
print("\n--- Example user message ---")
print(build_user_message(df.iloc[0]))

## Generate Descriptions

In [ ]:
def generate_description(row: pd.Series) -> dict:
    """Call the model and return description + metrics."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_message(row)},
    ]

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=200,
    )
    elapsed_ms = (time.perf_counter() - start) * 1000

    usage = response.usage
    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms": round(elapsed_ms, 1),
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
    }

In [ ]:
results = []

for idx, row in df.iterrows():
    result = generate_description(row)
    results.append(result)
    print(f"[{idx + 1}/{len(df)}] {row['product_name']}: "
          f"{result['latency_ms']:.0f}ms, "
          f"{len(result['generated_description'].split())} words")

print(f"\nDone. Generated {len(results)} descriptions.")

## Build Output DataFrame

In [ ]:
results_df = pd.DataFrame(results)
output_df = pd.concat([df, results_df], axis=1)

# Add blank evaluation columns
for criterion in CRITERIA:
    output_df[criterion] = ""
output_df["final_score"] = ""

print(f"Output shape: {output_df.shape}")
print(f"Columns: {list(output_df.columns)}")
output_df.head(3)

In [ ]:
OUTPUT_EXCEL.parent.mkdir(parents=True, exist_ok=True)
output_df.to_excel(OUTPUT_EXCEL, index=False)
print(f"Saved to {OUTPUT_EXCEL}")

## Quick Stats

In [ ]:
output_df["word_count"] = output_df["generated_description"].apply(lambda x: len(x.split()))

print("Latency (ms):")
print(output_df["latency_ms"].describe().round(1))
print()
print("Word count:")
print(output_df["word_count"].describe().round(1))
print()
in_range = output_df["word_count"].between(50, 90).sum()
print(f"Descriptions in 50-90 word range: {in_range}/{len(output_df)}")